# Platoon Ratio & Arrival Type from hi-res controller data

This notebook demonstrates the new **`platoon_ratio`** aggregation in the
[`atspm`](https://github.com/ShawnStrasser/atspm) Python package
(pull request from [Abimbola08/atspm-python](https://github.com/Abimbola08/atspm-python/tree/feature/platoon-ratio), closes upstream issue #6).

**What it computes** (HCM 7th ed., Chapter 19):

$$R_p = \frac{P}{g/C}$$

where $P$ is the proportion of vehicles arriving on green (`Percent_AOG` from the
`arrival_on_green` aggregation) and $g/C$ is the green ratio, estimated from the phase's
green time inside each aggregation bin.

| Arrival type | Platoon ratio $R_p$ | Progression quality |
|---|---|---|
| 1 | ≤ 0.50 | Very poor (dense platoon arrives at start of red) |
| 2 | 0.51 – 0.85 | Unfavorable |
| 3 | 0.86 – 1.15 | Random arrivals |
| 4 | 1.16 – 1.50 | Favorable |
| 5 | 1.51 – 2.00 | Highly favorable |
| 6 | > 2.00 | Exceptional (dense platoon arrives at start of green) |

**What you need:** hi-res event data from a controller (`TimeStamp, DeviceId, EventId, Parameter`)
and a detector configuration (`DeviceId, Phase, Parameter, Function`) identifying the *advance*
detectors. The package ships a 2-hour sample from one intersection, so you can run everything
below without any data of your own — then drop in your own files in the last section.


## 1. Install the package from the feature branch

In [ ]:
%pip install -q "git+https://github.com/Abimbola08/atspm-python.git@feature/platoon-ratio"
import atspm, duckdb, pandas as pd
print("atspm", atspm.__version__ if hasattr(atspm, "__version__") else "(dev)", "| duckdb", duckdb.__version__)

## 2. Look at the bundled hi-res sample

Standard Indiana/UDOT event codes used by the aggregation:

- `1` phase begin green, `8` phase begin yellow — used to build green intervals per phase
- `82` detector on — actuations, matched to phases through the detector config (`Function = 'Advance'`)

In [ ]:
from atspm import sample_data

raw = sample_data.data.df()
cfg = sample_data.config.df()

print(f"{len(raw):,} events from {raw.TimeStamp.min()} to {raw.TimeStamp.max()} | devices: {raw.DeviceId.unique()}")
display(raw.head())
print("Advance detectors (these drive arrival-on-green and platoon ratio):")
display(cfg[cfg.Function == "Advance"].sort_values("Phase"))

## 3. Run the aggregations

`platoon_ratio` depends on `arrival_on_green`; the processor orders them automatically via
`AGGREGATION_DEPENDENCIES`, but we list both here for clarity.

In [ ]:
from atspm import SignalDataProcessor

params = {
    "raw_data": raw,          # pandas DataFrame (a file path also works)
    "detector_config": cfg,
    "bin_size": 15,              # minutes
    "output_dir": "atspm_output",
    "output_format": "csv",
    "output_to_separate_folders": False,
    "remove_incomplete": False,
    "verbose": 1,
    "aggregations": [
        {"name": "arrival_on_green", "params": {"latency_offset_seconds": 0}},
        {"name": "platoon_ratio",    "params": {}},
    ],
}

processor = SignalDataProcessor(**params)
processor.load()
processor.aggregate()

aog = processor.conn.query("SELECT * FROM arrival_on_green ORDER BY Phase, TimeStamp").df()
rp  = processor.conn.query("SELECT * FROM platoon_ratio    ORDER BY Phase, TimeStamp").df()
processor.conn.close()

print(f"{len(rp)} phase-bins")
rp.head(12)

### Sanity checks

- `Green_Ratio` must be within (0, 1] (a phase can't be green more than 100 % of a bin).
- `Platoon_Ratio` equals `Percent_AOG / Green_Ratio` row by row.
- `Arrival_Type` follows the HCM thresholds.

In [ ]:
import numpy as np

assert rp.Green_Ratio.between(0, 1, inclusive="right").all()
assert np.allclose(rp.Platoon_Ratio, rp.Percent_AOG / rp.Green_Ratio, rtol=1e-4)
bins = [-np.inf, 0.50, 0.85, 1.15, 1.50, 2.00, np.inf]
assert (pd.cut(rp.Platoon_Ratio, bins, labels=[1, 2, 3, 4, 5, 6]).astype(int) == rp.Arrival_Type).all()
print("all checks passed")

rp.groupby("Phase").agg(bins=("TimeStamp", "size"),
                        mean_AOG=("Percent_AOG", "mean"),
                        mean_gC=("Green_Ratio", "mean"),
                        mean_Rp=("Platoon_Ratio", "mean"),
                        modal_AT=("Arrival_Type", lambda s: s.mode().iat[0])).round(3)

## 4. Why platoon ratio and not just arrival on green?

Percent arrival on green rewards phases that simply get a lot of green. Platoon ratio normalizes
by the green ratio, so a coordinated through phase (phase 2/6, big split) and a minor phase can
be compared on the same scale, and a value near 1 means arrivals are no better than random.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for ph, g in rp.groupby("Phase"):
    axes[0].plot(g.TimeStamp, g.Percent_AOG, marker="o", label=f"Phase {ph}")
    axes[1].plot(g.TimeStamp, g.Platoon_Ratio, marker="o", label=f"Phase {ph}")
axes[0].set_title("Percent arrival on green"); axes[0].set_ylim(0, 1)
axes[1].set_title("Platoon ratio  Rp = P / (g/C)")
for y, lbl in [(0.5, "AT1|2"), (0.85, "AT2|3"), (1.15, "AT3|4"), (1.5, "AT4|5"), (2.0, "AT5|6")]:
    axes[1].axhline(y, color="grey", lw=0.6, ls="--"); axes[1].text(rp.TimeStamp.min(), y, lbl, fontsize=7, va="bottom")
axes[1].legend(fontsize=8, ncol=2)
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Distribution of arrival types by phase
pd.crosstab(rp.Phase, rp.Arrival_Type).reindex(columns=range(1, 7), fill_value=0)

## 5. Try it on your own controller data

Upload two files (CSV or Parquet):

1. **Hi-res events** with columns `TimeStamp, DeviceId, EventId, Parameter`
   (the standard hi-res log format from MaxTime, Econolite, Intelight, etc.; only events 1, 8 and 82 matter here).
2. **Detector config** with columns `DeviceId, Phase, Parameter, Function`, where `Parameter` is the
   detector channel and `Function` is `Advance` for the setback/advance detectors used for arrival on green.

Then run the cell below. If you don't have the files handy, this cell just re-runs the sample.

In [ ]:
import os
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

raw_path, cfg_path = None, None
if IN_COLAB:
    print("Select your hi-res event file (or cancel to use the sample):")
    up = files.upload()
    raw_path = next(iter(up), None)
    if raw_path:
        print("Now select your detector config file:")
        up = files.upload()
        cfg_path = next(iter(up), None)

if raw_path and cfg_path:
    my_params = {**params, "raw_data": raw_path, "detector_config": cfg_path,
                 "output_dir": "my_output"}
else:
    print("Using bundled sample data.")
    my_params = params

with SignalDataProcessor(**my_params) as p:
    p.load(); p.aggregate()
    my_rp = p.conn.query("SELECT * FROM platoon_ratio ORDER BY DeviceId, Phase, TimeStamp").df()

my_rp.to_csv("platoon_ratio.csv", index=False)
print(f"{len(my_rp)} rows written to platoon_ratio.csv")
my_rp.head(20)

## Notes on the implementation

- Green intervals are built per phase from event 1 (begin green) to the next event 8 (begin yellow), split across
  bin boundaries so a 15-minute bin only counts the green seconds that fall inside it.
- A green still open at the end of the data is extended to the end of its bin; a yellow with no preceding
  green (data starts mid-green) is assumed green from the start of its bin. These rules make the result
  identical whether the data is processed in one pass or incrementally in 15-minute chunks.
- Bins with zero green time for a phase are dropped rather than divided by zero.
- Source: [`src/atspm/queries/platoon_ratio.sql`](https://github.com/Abimbola08/atspm-python/blob/feature/platoon-ratio/src/atspm/queries/platoon_ratio.sql)
